# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/OjaswiGautam/FlyrankAI/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

Per the training-honest-models skill file's own method-selection table — "grouping items → K-Means (pick k with silhouette), then NAME clusters after inspecting them → unsupervised needs human naming" — K-Means is the directly recommended method for this exact question shape, and it matches Lane 3's task framing established back in w02: this is unsupervised archetype discovery, not classification or ranking, so there is no label to fit against and no precision@K to chase.

Why K-Means specifically, over the toolkit's other options:

*   Not Logistic Regression / Random Forest / Gradient Boosting — these require a supervised label. Lane 3 has none, by design (w02 established this explicitly: inventing a proxy label just to use a supervised method would mean fabricating the exact "ground truth" this lane is supposed to discover, not assume).

*   Not correlation/signal analysis alone — useful as a diagnostic (and we used it in w04's signal checks), but it answers "does X relate to Y," not "what groups exist." It doesn't produce archetypes.

*   K-Means over other clustering methods (e.g., GMM, HDBSCAN) — K-Means is the method the skill file names directly for this task shape, it produces hard, interpretable cluster assignments (a page belongs to exactly one archetype, which maps cleanly to a review workflow), and it lets us pick k transparently via silhouette rather than relying on a density parameter that's harder to justify to a non-technical reviewer. Alternative methods remain a reasonable future extension (noted as a stretch goal from the recent review), not a requirement for this baseline model.

*   Fits the feature frame directly — the 5-feature frame from w03 (gsc_impressions, gsc_avg_position, gsc_clicks, content_age_days, word_count), all numeric after preprocessing (log-transform, imputation, missingness flags, scaling), is exactly the kind of continuous feature space K-Means is built for — a distance-based method needs numeric, scaled inputs, which this frame provides after the preprocessing work already done and verified.

What this method can and cannot do, stated up front: K-Means will produce a fixed number of behavioral groups based on distance in the scaled feature space — it does not predict, does not rank by priority, and does not explain why a page behaves a certain way. It answers "what recurring patterns exist," consistent with the lane's original research question from w01, not "what should be done about it" — that interpretive step happens afterward, by inspecting real cluster profiles and examples, not by the algorithm itself.





## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

Per the lane guide's Validation Rules section — "client/group holdout, when pages from the same client may share patterns the model could memorize" — this project uses a client-grouped split: GroupShuffleSplit on client_hash_id, 75/25, producing 35 train clients / 133,474 rows and 12 validation clients / 43,264 rows, with zero client overlap between the two sets (verified directly in output, not assumed).

Why grouped-by-client, over the toolkit's other split options:

*   Not a plain random row split — pages from the same client are not independent observations. A single client's content likely shares a CMS template, an editorial word-count convention, and a consistent GSC tracking setup, so pages from the same client will naturally sit close together in feature space regardless of any real archetype. A random split would let near-duplicate client patterns leak into both train and validation, inflating the silhouette and Davies-Bouldin scores by rewarding the model for memorizing a client's "house style" rather than discovering a genuine cross-client behavioral pattern. This is the exact failure mode the leakage checklist warns against: "are duplicate or related rows split across train and test in a way that makes the test too easy?"

*  Not a time-aware split — a time-aware train/test split is the right design when the question is about predicting a future outcome from a prior window (the lane guide reserves this explicitly for the Growth/Recovery/Momentum freestyle direction). This notebook's population is a single fixed window — March 2026, gsc_data_available=TRUE — aggregated to one row per (client, content) with no forward-looking target. Lane 3's question, established in w01/w02, is "what recurring behavioral archetypes exist," not "what will happen next" — there is no future window to hold out, so a time split would be solving a problem this lane doesn't have.

*  Client-grouped is what the modeling population's own structure demands — the sanity checks already run in this notebook (Cell 14's per-cluster top-client-share numbers: 14.0%–39.3%) confirm that client identity is a real, measurable source of correlation within the feature space. A validation design that ignores this would be validating against noise it already knows exists.

What this split can and cannot prove: a client-grouped holdout tells us whether the cluster structure generalizes to clients the model has never seen — which is the right bar for an archetype system meant to apply across FlyRank's whole client base, not just the training clients. It does not tell us whether the structure is stable over time for a given client (that would need a time-aware design layered on top, and is out of scope for this single-month, cross-sectional clustering question). The val-set silhouette (0.3568) and Davies-Bouldin (0.9087) reported for k=4 are therefore honestly comparable to the train-set numbers precisely because the split guarantees no client-level shortcut was available to either side.





## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

The model uses the same client-grouped population and modeling frame established in the model-build notebook, preprocessed with StandardScaler (not RobustScaler) and the original, justified 5-feature set from w03 — no added collinear features like gsc_ctr (which is mathematically derived from two features already in the matrix and would distort distance calculations if included separately).

After sweeping k=3–8 with a full validation stack (silhouette, Davies-Bouldin, Calinski-Harabasz, cluster balance) on a client-grouped 75/25 split, k=4 was selected on real evidence: best validation silhouette (0.3568) and best validation Davies-Bouldin (0.9087), later confirmed stable across 5 random seeds (mean ARI 0.9985, after catching and fixing an n_init=10 local-optimum bug on one seed).

Because Lane 3 is unsupervised and the Week 4 baseline has no precision@K (no label exists), silhouette and baseline coverage are not directly comparable metrics. Instead, per the workflow's own guidance, the Week 4 baseline (TITLE_META_CTR_FIX) is rebuilt exactly and overlaid onto the fitted clusters as a post-hoc diagnostic, never as a model input.

In [4]:
import os
import duckdb
import numpy as np
import pandas as pd
from google.colab import userdata
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupShuffleSplit
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score, adjusted_rand_score

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
con = duckdb.connect()
con.sql("INSTALL httpfs; LOAD httpfs;")
con.sql(f"CREATE SECRET hf_token (TYPE HUGGINGFACE, TOKEN '{os.environ['HF_TOKEN']}');")

TABLE = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet"
DIM = "hf://datasets/FlyRank/internship-warehouse/dim_content.parquet"

# --- 1. Build modeling frame, with a real, verified date-window check ---
raw = con.sql(f"""
    WITH scoped AS (
        SELECT client_hash_id, content_hash_id, report_date, gsc_impressions, gsc_clicks, gsc_avg_position
        FROM read_parquet('{TABLE}')
        WHERE gsc_data_available = TRUE
    ),
    agg AS (
        SELECT client_hash_id, content_hash_id,
               MIN(report_date) AS min_date, MAX(report_date) AS max_date,
               COUNT(DISTINCT report_date) AS distinct_days,
               SUM(gsc_impressions) AS gsc_impressions,
               SUM(gsc_clicks) AS gsc_clicks,
               SUM(gsc_impressions * gsc_avg_position) / NULLIF(SUM(gsc_impressions), 0) AS gsc_avg_position
        FROM scoped
        GROUP BY client_hash_id, content_hash_id
    )
    SELECT a.*, d.content_created_date, d.word_count
    FROM agg a
    LEFT JOIN read_parquet('{DIM}') d
      ON a.client_hash_id = d.client_hash_id
     AND a.content_hash_id = d.content_hash_id
    ORDER BY a.client_hash_id, a.content_hash_id
""").df()

# Real, executed window check — not a silent skip
print("Observation window check:")
print("  min_date:", raw['min_date'].min(), "| max_date:", raw['max_date'].max())
assert raw['min_date'].min() >= pd.Timestamp('2026-03-01') and raw['max_date'].max() <= pd.Timestamp('2026-03-31'), \
    "Window check failed — data outside March 2026"
print("  PASS — window confirmed within March 2026")

df = raw.copy()
df['content_age_days'] = (pd.Timestamp('2026-03-31') - pd.to_datetime(df['content_created_date'])).dt.days

# Duplicate-key check
dupes = df.duplicated(['client_hash_id', 'content_hash_id']).sum()
print(f"\nDuplicate (client, content) keys: {dupes}")
assert dupes == 0, "Grain check failed"

# --- 2. Leakage check ---
forbidden_fields = {"health_score", "priority_score", "action_type", "action_label",
                     "reason_code", "cluster", "archetype", "recommendation"}
leaked = forbidden_fields.intersection(df.columns)
print("Forbidden fields present:", leaked or "NONE — clean")

# --- 3. Preprocessing (matches w03/w04-justified feature set, StandardScaler) ---
df['avg_position_missing_or_zero'] = (df['gsc_avg_position'] == 0).astype(int)
df['avg_position_clean'] = df['gsc_avg_position'].replace(0, np.nan)
avg_position_median = df.loc[df['avg_position_clean'].notna(), 'avg_position_clean'].median()
df['avg_position_clean'] = df['avg_position_clean'].fillna(avg_position_median)

df['log_gsc_impressions'] = np.log1p(df['gsc_impressions'])
df['log_gsc_clicks'] = np.log1p(df['gsc_clicks'])

df['word_count_missing'] = df['word_count'].isna().astype(int)
word_count_median = df['word_count'].median()
df['word_count'] = df['word_count'].fillna(word_count_median)

MODEL_FEATURES = ['log_gsc_impressions', 'avg_position_clean', 'log_gsc_clicks',
                   'content_age_days', 'word_count',
                   'avg_position_missing_or_zero', 'word_count_missing']

# --- 4. Client-grouped split ---
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, val_idx = next(gss.split(df, groups=df['client_hash_id'].values))
train_df = df.iloc[train_idx].reset_index(drop=True)
val_df = df.iloc[val_idx].reset_index(drop=True)

scaler = StandardScaler()
X_train = scaler.fit_transform(train_df[MODEL_FEATURES])
X_val = scaler.transform(val_df[MODEL_FEATURES])

print(f"\nTrain: {len(train_df)} rows, {train_df['client_hash_id'].nunique()} clients")
print(f"Val: {len(val_df)} rows, {val_df['client_hash_id'].nunique()} clients")
print("Client overlap:", len(set(train_df['client_hash_id']) & set(val_df['client_hash_id'])))

# --- 5. k-selection sweep ---
k_rows = []
for k in range(3, 9):
    km = KMeans(n_clusters=k, random_state=42, n_init=30)
    tl = km.fit_predict(X_train)
    vl = km.predict(X_val)
    k_rows.append({
        'k': k,
        'val_silhouette': round(silhouette_score(X_val, vl, sample_size=20000, random_state=42), 4),
        'val_davies_bouldin': round(davies_bouldin_score(X_val, vl), 4),
        'min_val_cluster_share': round(pd.Series(vl).value_counts(normalize=True).min(), 4),
    })
k_df = pd.DataFrame(k_rows)
print("\nK-selection:\n", k_df.to_string(index=False))

# --- 6. Final model, k=4 ---
FINAL_K = 4
final_km = KMeans(n_clusters=FINAL_K, random_state=42, n_init=30)
train_labels = final_km.fit_predict(X_train)
val_labels = final_km.predict(X_val)
train_df['cluster'] = train_labels
val_df['cluster'] = val_labels
full_labeled = pd.concat([train_df.assign(split='train'), val_df.assign(split='val')], ignore_index=True)

# --- 7. Rebuild Week 4 baseline exactly, verify it matches ---
baseline_base = full_labeled[(full_labeled['gsc_impressions'] >= 100) & (full_labeled['gsc_avg_position'] > 0)].copy()
baseline_base['ctr'] = baseline_base['gsc_clicks'] / baseline_base['gsc_impressions']

def position_tier(pos):
    if pos <= 3: return 'pos_1_3'
    elif pos <= 10: return 'pos_4_10'
    elif pos <= 20: return 'pos_11_20'
    else: return 'pos_21_plus'

baseline_base['position_tier'] = baseline_base['gsc_avg_position'].apply(position_tier)

tier_ctr = baseline_base.groupby('position_tier').apply(
    lambda g: g['gsc_clicks'].sum() / g['gsc_impressions'].sum(), include_groups=False
).to_dict()
baseline_base['expected_ctr'] = baseline_base['position_tier'].map(tier_ctr)
baseline_base['ctr_gap_pct'] = (baseline_base['expected_ctr'] - baseline_base['ctr']).clip(lower=0) / baseline_base['expected_ctr']
baseline_base['baseline_flag'] = (baseline_base['ctr_gap_pct'] >= 0.30).astype(int)

print(f"\nRebuilt baseline queue: {baseline_base['baseline_flag'].sum()} rows (expect 61,267)")

# --- 7b. MERGE the baseline flag back onto full_labeled (this was missing) ---
if 'baseline_flag' in full_labeled.columns:
    full_labeled = full_labeled.drop(columns=['baseline_flag'])

full_labeled = full_labeled.merge(
    baseline_base[['client_hash_id', 'content_hash_id', 'baseline_flag']],
    on=['client_hash_id', 'content_hash_id'], how='left'
)
full_labeled['baseline_flag'] = full_labeled['baseline_flag'].fillna(0).astype(int)

print("Baseline flag merged onto full_labeled. Total flagged:", full_labeled['baseline_flag'].sum())

# --- 8. Baseline overlay by cluster ---
global_rate = full_labeled['baseline_flag'].mean()
lift = full_labeled.groupby('cluster').agg(
    cluster_rows=('cluster', 'size'),
    baseline_flagged=('baseline_flag', 'sum'),
).reset_index()
lift['flag_rate'] = (lift['baseline_flagged'] / lift['cluster_rows']).round(4)
lift['lift'] = (lift['flag_rate'] / global_rate).round(3)
print("\nBaseline lift by cluster:\n", lift.to_string(index=False))

# --- 8. Baseline overlay by cluster ---
global_rate = full_labeled['baseline_flag'].mean()
lift = full_labeled.groupby('cluster').agg(
    cluster_rows=('cluster', 'size'),
    baseline_flagged=('baseline_flag', 'sum'),
).reset_index()
lift['flag_rate'] = (lift['baseline_flagged'] / lift['cluster_rows']).round(4)
lift['lift'] = (lift['flag_rate'] / global_rate).round(3)
print("\nBaseline lift by cluster:\n", lift.to_string(index=False))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Observation window check:
  min_date: 2026-03-01 00:00:00 | max_date: 2026-03-31 00:00:00
  PASS — window confirmed within March 2026

Duplicate (client, content) keys: 0
Forbidden fields present: NONE — clean

Train: 133474 rows, 35 clients
Val: 43264 rows, 12 clients
Client overlap: 0

K-selection:
  k  val_silhouette  val_davies_bouldin  min_val_cluster_share
 3          0.3485              1.1340                 0.2649
 4          0.3568              0.9087                 0.0054
 5          0.3416              1.1002                 0.0054
 6          0.3395              0.9820                 0.0054
 7          0.3314              1.0163                 0.0044
 8          0.3270              0.9931                 0.0044

Rebuilt baseline queue: 61267 rows (expect 61,267)
Baseline flag merged onto full_labeled. Total flagged: 61267

Baseline lift by cluster:
  cluster  cluster_rows  baseline_flagged  flag_rate  lift
       0         52572             20234     0.3849 1.110
      

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

Unsupervised models like K-Means can hide their true nature behind aggregate metrics. A validation silhouette of 0.3568 tells us the clusters have moderate geometric separation in the scaled feature space — it does not prove business relevance or "correctness," since there's no ground truth to check against. Below, we stress-test the model directly: which rows sit assigned to the lowest-traffic cluster despite having genuinely high impressions, and what does the centroid-distance math say about why?

Code (rewritten against the corrected k=4 pipeline — full_labeled, final_km, MODEL_FEATURES, scaler):

In [5]:
import numpy as np
import pandas as pd

required_objects = ["full_labeled", "scaler", "MODEL_FEATURES", "final_km"]
missing_objects = [name for name in required_objects if name not in globals()]

if missing_objects:
    print("Run Section 3's modeling cell first so these objects exist: " + ", ".join(missing_objects))
else:
    section4_df = full_labeled.copy()

    X_all = scaler.transform(section4_df[MODEL_FEATURES])
    centroids = final_km.cluster_centers_
    labels = section4_df["cluster"].to_numpy()

    # Squared Euclidean distance to every centroid: rows x clusters
    diff = X_all[:, None, :] - centroids[None, :, :]
    sq_contrib = diff ** 2
    distances = sq_contrib.sum(axis=2)

    assigned_distance = distances[np.arange(len(section4_df)), labels]
    masked_distances = distances.copy()
    masked_distances[np.arange(len(section4_df)), labels] = np.inf
    nearest_alt_cluster = masked_distances.argmin(axis=1)
    nearest_alt_distance = masked_distances[np.arange(len(section4_df)), nearest_alt_cluster]
    distance_margin = nearest_alt_distance - assigned_distance

    section4_df["assigned_distance"] = assigned_distance
    section4_df["nearest_alt_cluster"] = nearest_alt_cluster
    section4_df["distance_margin"] = distance_margin

    # Cluster profile receipts, real k=4 model
    profile_rows = []
    for cluster_value, cluster_frame in section4_df.groupby("cluster"):
        row = {
            "cluster_id": cluster_value,
            "rows": len(cluster_frame),
            "row_share": round(len(cluster_frame) / len(section4_df), 4),
            "clients": cluster_frame["client_hash_id"].nunique(),
            "top_client_share": round(cluster_frame["client_hash_id"].value_counts(normalize=True).iloc[0], 4),
            "gsc_impressions_median": cluster_frame["gsc_impressions"].median(),
            "gsc_clicks_median": cluster_frame["gsc_clicks"].median(),
        }
        profile_rows.append(row)
    cluster_profile = pd.DataFrame(profile_rows).sort_values("cluster_id")
    print("Cluster profile receipts (k=4, real balanced result):")
    print(cluster_profile.to_string(index=False))
    print("\n" + "="*80 + "\n")

    # Tension-case stress test: rows in the LOWEST-median-impression cluster
    # that are still at/above the 90th percentile of impressions overall
    impression_medians = section4_df.groupby("cluster")["gsc_impressions"].median().sort_values()
    low_impression_cluster = impression_medians.index[0]
    high_impression_cluster = impression_medians.index[-1]
    high_impression_threshold = section4_df["gsc_impressions"].quantile(0.90)

    tension_mask = (
        (section4_df["cluster"] == low_impression_cluster)
        & (section4_df["gsc_impressions"] >= high_impression_threshold)
    )
    tension_cases = section4_df.loc[tension_mask].copy()

    print(f"Tension-case definition: rows in cluster {low_impression_cluster} (lowest median impressions)")
    print(f"but at/above the 90th percentile of impressions ({high_impression_threshold:.0f}).")
    print(f"Tension case rows found: {len(tension_cases)}\n")

    if len(tension_cases) > 0:
        MODEL_FEATURE_NAMES = MODEL_FEATURES  # scaler has no get_feature_names_out; use the known list directly
        max_cases = min(3, len(tension_cases))
        contribution_rows = []
        for idx in tension_cases.index[:max_cases]:
            pos = section4_df.index.get_loc(idx)
            assigned_c = int(labels[pos])
            nearest_c = int(nearest_alt_cluster[pos])
            delta = sq_contrib[pos, nearest_c, :] - sq_contrib[pos, assigned_c, :]
            top_features = np.argsort(np.abs(delta))[::-1][:3]

            contribution_rows.append({
                "content_hash_id": section4_df.loc[idx, "content_hash_id"][:14] + "...",
                "client_hash_id": section4_df.loc[idx, "client_hash_id"][:14] + "...",
                "assigned_cluster": assigned_c,
                "nearest_alt_cluster": nearest_c,
                "distance_margin": round(float(distance_margin[pos]), 3),
                "gsc_impressions": section4_df.loc[idx, "gsc_impressions"],
                "gsc_clicks": section4_df.loc[idx, "gsc_clicks"],
                "top_driving_features": [MODEL_FEATURE_NAMES[i] for i in top_features],
            })
        print(pd.DataFrame(contribution_rows).to_string(index=False))
    else:
        print("No tension cases found under this definition — see interpretation note below.")

    print("\n" + "="*80)
    print("Interpretation: a small distance_margin means the row sits near a cluster boundary —")
    print("close to being assigned to the alternative cluster instead. Feature deltas show which")
    print("scaled inputs contributed most to the distance difference; they explain the geometry,")
    print("not a causal reason the page 'belongs' to either group.")

Cluster profile receipts (k=4, real balanced result):
 cluster_id  rows  row_share  clients  top_client_share  gsc_impressions_median  gsc_clicks_median
          0 52572     0.2975       23            0.3142                    96.0                0.0
          1 48766     0.2759       33            0.2270                  2533.0                6.0
          2 73966     0.4185       47            0.1403                    38.0                0.0
          3  1434     0.0081       30            0.3926                     1.0                0.0


Tension-case definition: rows in cluster 3 (lowest median impressions)
but at/above the 90th percentile of impressions (3930).
Tension case rows found: 0

No tension cases found under this definition — see interpretation note below.

Interpretation: a small distance_margin means the row sits near a cluster boundary —
close to being assigned to the alternative cluster instead. Feature deltas show which
scaled inputs contributed most to the distan

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.